In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
import pypulseq as pp
import MRzeroCore as mr0
import torchkbnufft as tkbn
import sys
sys.path.insert(0, "..")
from Modules import load_brain_phantom, default_system
torch.set_default_device("cuda")

In [ ]:
def mpnrage_seq(
    fov: float = 250e-3,
    Nx: int = 128,
    alpha: float = 5.0,
    slice_thickness: float = 5e-3,
    TE: float = 3.98e-3,
    N_readout: int = 50,
    N_spokes: int = 256,
    N_dummy: int = 20,
    TI_start: float = 20e-3,
    TI_inter: float = 100e-3,
    TD: float = 1000e-3,
    rf_spoiling_inc: float = 117.0,
    write_seq: bool = False,
) -> pp.Sequence:

    system = default_system()
    seq = pp.Sequence(system)
    deltak = 1 / fov

    rf_inv, gz_inv, _ = pp.make_sinc_pulse(
        flip_angle=np.pi, duration=10e-3, slice_thickness=slice_thickness,
        apodization=0.5, time_bw_product=4, system=system,
        return_gz=True, use='inversion')

    rf, gz, _ = pp.make_sinc_pulse(
        apodization=0.5, duration=2e-3, flip_angle=alpha * np.pi / 180,
        slice_thickness=slice_thickness, system=system,
        time_bw_product=4, return_gz=True, use='excitation')

    gx_base = pp.make_trapezoid(channel='x', flat_area=Nx * deltak,
                                flat_time=6.4e-3 / 5, system=system)
    adc = pp.make_adc(num_samples=Nx, duration=gx_base.flat_time,
                      delay=gx_base.rise_time, system=system)

    gx_pre_base = pp.make_trapezoid(channel='x', area=-gx_base.area / 2,
                                    duration=2e-3, system=system)
    gz_reph     = pp.make_trapezoid(channel='z', area=-gz.area / 2,
                                    duration=2e-3, system=system)
    gx_spoil_base = pp.make_trapezoid(channel='x', area=0.5 * Nx * deltak,
                                      system=system)
    gz_spoil    = pp.make_trapezoid(channel='z', area=4 / slice_thickness,
                                    system=system)
 
    delay_TE = (
        np.ceil((
            TE
            - pp.calc_duration(gx_pre_base)
            - gz.fall_time - gz.flat_time / 2
            - pp.calc_duration(gx_base) / 2
        ) / seq.grad_raster_time) * seq.grad_raster_time
    )
    inv_duration   = pp.calc_duration(rf_inv, gz_inv)
    delay_TI_start = (
        np.ceil((TI_start - inv_duration / 2 - pp.calc_duration(gz) / 2)
                / seq.grad_raster_time) * seq.grad_raster_time
    )
    assert delay_TI_start > 0, 'TI_start too short'

    gre_block_dur = (
        pp.calc_duration(gz)
        + pp.calc_duration(gx_pre_base)
        + delay_TE
        + pp.calc_duration(gx_base)
        + pp.calc_duration(gx_spoil_base, gz_spoil)
    )
    delay_TD = np.ceil(TD / seq.grad_raster_time) * seq.grad_raster_time
    delay_TI_inter = np.ceil(TI_inter / seq.grad_raster_time) * seq.grad_raster_time

    GA_DEG = 111.246  
    rf_phase = 0.0; rf_inc = 0.0; spoke_counter = 0

    for seg in range(N_spokes):
        seq.add_block(rf_inv, gz_inv)
        seq.add_block(pp.make_delay(delay_TI_start))

        n_this_seg = N_readout + (N_dummy if seg == 0 else 0)
        for s in range(n_this_seg):
            is_dummy = (seg == 0) and (s < N_dummy)

            rf.phase_offset  = rf_phase / 180 * np.pi
            adc.phase_offset = rf_phase / 180 * np.pi
            rf_inc   = (rf_inc + rf_spoiling_inc) % 360
            rf_phase = (rf_phase + rf_inc) % 360

            if is_dummy:
                angle = 0.0
            else:
                angle = np.deg2rad(spoke_counter * GA_DEG)
                spoke_counter += 1

            kx = np.cos(angle)  
            ky = np.sin(angle)

            seq.add_block(rf, gz)
            seq.add_block(
                pp.make_trapezoid(channel='x', area=gx_pre_base.area * kx,
                                  duration=2e-3, system=system),
                pp.make_trapezoid(channel='y', area=gx_pre_base.area * ky,
                                  duration=2e-3, system=system),
                gz_reph,  
            )
            seq.add_block(pp.make_delay(delay_TE))

            gx = pp.make_trapezoid(channel='x',
                                   amplitude=gx_base.amplitude * kx,
                                   flat_time=gx_base.flat_time,
                                   rise_time=gx_base.rise_time,
                                   fall_time=gx_base.fall_time,
                                   system=system)
            gy = pp.make_trapezoid(channel='y',
                                   amplitude=gx_base.amplitude * ky,
                                   flat_time=gx_base.flat_time,
                                   rise_time=gx_base.rise_time,
                                   fall_time=gx_base.fall_time,
                                   system=system)

            if is_dummy:
                seq.add_block(gx, gy)
            else:
                seq.add_block(gx, gy, adc)

            seq.add_block(
                pp.make_trapezoid(channel='x', area=gx_spoil_base.area * kx,
                                  system=system),
                pp.make_trapezoid(channel='y', area=gx_spoil_base.area * ky,
                                  system=system),
                gz_spoil,
            )

            if s < n_this_seg - 1:
                seq.add_block(pp.make_delay(delay_TI_inter))

        seq.add_block(pp.make_delay(delay_TD))

    ok, err = seq.check_timing()
    if ok:
        print('2-D MPnRAGE timing check passed')
        print(f'  GRE block dur : {gre_block_dur * 1000:.1f} ms')
        print(f'  Readout train : {N_readout * gre_block_dur * 1000:.0f} ms')
        est = N_spokes * (inv_duration + delay_TI_start
                          + N_readout * gre_block_dur + TD)
        print(f'  Scan time est : {est / 60:.1f} min')
    else:
        print(f'Timing FAILED: {err}')

    seq.set_definition('FOV',  [fov, fov, slice_thickness])
    seq.set_definition('Name', 'mpnrage_2d_radial')
    if write_seq:
        seq.write('MPnRAGE_2D_Radial.seq')
    return seq, gre_block_dur, TI_start

In [ ]:
device = torch.device("cuda")
fov: float = 250e-3,
Nx: int = 128,
alpha: float = 5.0,
slice_thickness: float = 5e-3,
TE: float = 3.98e-3,
N_readout: int = 50,
N_spokes: int = 256,
N_dummy: int = 20,
TI_start: float = 20e-3,
TI_inter: float = 100e-3,
TD: float = 1000e-3,
rf_spoiling_inc: float = 117.0,

In [ ]:
seq, gre_block_dur, TI_start = mpnrage_seq(
    write_seq=True, 
    fov=fov,
    Nx=Nx,
    alpha=alpha,
    slice_thickness=slice_thickness,
    TE=TE,
    N_readout=N_readout, 
    N_spokes=N_spokes,
    N_dummy=N_dummy,
    TI_start=TI_start,
    TI_inter=TI_inter,
    TD=TD,
    rf_spoiling_inc=rf_spoiling_inc
)

In [ ]:
phantom = mr0.util.load_phantom(size=(N_x, N_x))

coil_sensitivities_recon = cyclic_head_coil((N_channels, N_x, N_x, 1)).to(
    torch.complex64
)
coil_sensitivities_recon = coil_sensitivities_recon.flip((1, 2))
coil_sensitivities_recon = coil_sensitivities_recon[..., 0]

coil_sensitivities_sim = cyclic_head_coil(
    (N_channels, N_x, N_x, 1),
    fov_scaling=(phantom.size[0] / FOV, phantom.size[1] / FOV, 1),
).to(torch.complex64)
phantom.coil_sens = coil_sensitivities_sim.to(device)
phantom.B0 = 0 * phantom.B0
phantom.B1 = torch.ones(phantom.B1.shape, dtype=phantom.B1.dtype)

data = phantom.build()

In [ ]:
mr0_seq = mr0.Sequence.import_file("MPnRAGE_2D_Radial.seq")
torch.manual_seed(0)
signal = isochromat_sim_fast(
    mr0_seq, data.cuda(), spin_count=256, spin_dist="rand", print_progress=True,
)
signal_all = signal.reshape(N_spokes, N_readout, Nx)
print(f"simulated signal: {tuple(signal_all.shape)}")

In [ ]:
def reconstruct_frame(
    signal_all, seq_file, frame_p, omega,
    N_readout, N_spokes, Nx_img, Ny_img, fov, device):

    deltak = 1 / fov
    omega  = min(omega, N_readout - frame_p)

    mr_seq      = mr0.Sequence.import_file(seq_file)
    ktraj_full  = torch.tensor(mr_seq.get_kspace(), dtype=torch.float32)
    ktraj_full  = ktraj_full.reshape(-1, ktraj_full.shape[-1])[:, :2]  

    sig_list, ktraj_list = [], []
    for seg in range(N_spokes):
        for s_off in range(omega):
            n     = frame_p + s_off
            start = (seg * N_readout + n) * Nx_img
            ktraj_list.append(ktraj_full[start : start + Nx_img])
            sig_list.append(signal_all[seg, n, :])

    sig    = torch.cat(sig_list, dim=0).to(torch.complex64)
    ktraj  = torch.cat(ktraj_list, dim=0)

    ktraj = ktraj / (deltak * Nx_img / 2) * np.pi * 0.85
    ktraj = ktraj.clamp(-np.pi, np.pi)
    ktraj = ktraj.T.contiguous().unsqueeze(0).to(device)  

    r      = torch.linalg.norm(ktraj[0], dim=0)
    dcomp  = r.view(1, 1, -1)

    sig        = sig.view(1, 1, -1).to(device)
    nufft_ob   = tkbn.KbNufftAdjoint(
        im_size=(Nx_img, Ny_img), grid_size=(2 * Nx_img, 2 * Ny_img)
    ).to(device)
    image = nufft_ob(sig * dcomp, ktraj)[0, 0]
    return torch.abs(image).cpu().numpy()


def ti_eff(frame_p, omega, TI_start, gre_block_dur, TI_inter=100e-3):
    centre = frame_p + (omega - 1) / 2
    return TI_start + centre * (gre_block_dur + TI_inter)

In [ ]:
omega    = 1
N_frames = N_readout - omega + 1

images, ti_effs = [], []
for p in range(N_frames):
    img   = reconstruct_frame(
        signal_all, 'MPnRAGE_2D_Radial.seq', frame_p=p, omega=omega,
        N_readout=N_readout, N_spokes=N_spokes,
        Nx_img=Nx_img, Ny_img=Ny_img, fov=fov, device=device)
    ti_ms = ti_eff(p, omega, TI_start, gre_block_dur) * 1000
    images.append(np.rot90(img, k=3))
    ti_effs.append(ti_ms)
    if p % 50 == 0:
        print(f'Frame {p:3d}/{N_frames}  TI_eff={ti_ms:.0f} ms')

print(f'Done -- {len(images)} frames, TI_eff {ti_effs[0]:.0f} -> {ti_effs[-1]:.0f} ms')

In [ ]:
def find_frame(tgt):
    return int(np.argmin(np.abs(np.array(ti_effs)-tgt)))

#fig, axes = plt.subplots(1, N_frames - 25, figsize=(N_frames - 25, 4))
fig, axes = plt.subplots(5, 5, figsize=(10, 10))
for p, ax in enumerate(axes.ravel()):
    ax.imshow(images[p], cmap='gray')
    ax.set_title(f'Frame {p}\nTI={ti_effs[p]:.0f}ms', fontsize=9)
    ax.axis('off')
plt.suptitle(f'MPnRAGE — {N_frames} frames', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
kspace = seq.calculate_kspace()
k_traj_adc = kspace[0]
k_traj_adc = k_traj_adc[:2, :]
k_traj_radius = np.sqrt(np.sum(k_traj_adc**2, axis=0, keepdims=True))
k_traj_adc = k_traj_adc / k_traj_radius.max() * 0.5
k_traj_adc = torch.tensor(k_traj_adc, dtype=torch.float32)

In [ ]:
import scipy
from scipy.optimize import root
scipy.__version__

import numpy as np
from scipy.optimize import root

idx1 = 5   
idx2 = 45  

ti_s = np.array(ti_effs) / 1000.0 
TI1 = ti_s[idx1]
TI2 = ti_s[idx2]

print(f"Using Frame {idx1} (TI = {TI1*1000:.0f} ms) and Frame {idx2} (TI = {TI2*1000:.0f} ms)")

img_stack = np.stack(images, axis=0)   
img1 = img_stack[idx1, :, :]
img2 = img_stack[idx2, :, :]

T1_map = np.zeros((Nx_img, Ny_img))

mean_img = img_stack.mean(axis=0)
mask = mean_img > 0.15 * mean_img.max()

def T1_from_two_points(S1, S2, TI1, TI2, T1_init=1.5):

    if S2 <= 0:
        return 0.0
    
    ratio = S1 / S2

    def f(T1):
        return (1 - 2*np.exp(-TI1/T1)) / (1 - 2*np.exp(-TI2/T1)) - ratio

    sol = root(lambda x: f(x[0]), x0=[T1_init])
    
    if sol.success and sol.x[0] > 0:
        return sol.x[0] * 1000  
    return 0.0

for ix in range(Nx_img):
    for iy in range(Ny_img):
        if not mask[ix, iy]:
            continue

        S1 = -img1[ix, iy]  
        S2 = img2[ix, iy]   
        
        T1_map[ix, iy] = T1_from_two_points(S1, S2, TI1, TI2)

T1_map[(T1_map > 5000) | (T1_map < 100)] = 0
T1_map[~mask] = 0

In [ ]:
plt.imshow(T1_map, cmap='magma')
plt.axis('off')
plt.colorbar()
plt.show()